In [ ]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

# IO.watch — Live Folder Monitoring

Watch a folder for incoming microscopy data. As new files are written (e.g. by Velox during acquisition), they automatically appear in the widget gallery — no need to re-run any cells.

This notebook simulates that workflow by writing synthetic TEM images to a temp folder on a timer.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import tempfile
import threading
import time
import numpy as np
import torch
from quantem.widget import IO, Show2D, Show3D, profile
profile()

## Generate synthetic TEM images

Create a helper that generates crystal lattice images with varying defocus, simulating a focal series captured on Velox. Each image is a different "capture" with slightly different Fresnel fringe contrast.

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def make_lattice_image(size=512, defocus=0.0, seed=42):
    """Generate a synthetic HRTEM crystal lattice image with defocus variation."""
    rng = torch.Generator(device="cpu").manual_seed(seed)
    y = torch.linspace(-5, 5, size, device=device)
    x = torch.linspace(-5, 5, size, device=device)
    Y, X = torch.meshgrid(y, x, indexing="ij")
    # Hexagonal lattice (graphene-like)
    a1 = torch.sin(2 * np.pi * (X + 0.5 * Y) * 2.5)
    a2 = torch.sin(2 * np.pi * Y * 2.5 * np.sqrt(3) / 2)
    lattice = (a1 + a2) ** 2
    # Defocus blur: Gaussian envelope that softens lattice fringes
    sigma = 0.3 + abs(defocus) * 0.15
    blur = torch.exp(-(X**2 + Y**2) / (2 * sigma**2))
    image = lattice * (0.3 + 0.7 * blur)
    # Add Poisson-like noise (on CPU for MPS compatibility)
    noise = torch.randn(size, size, generator=rng) * 0.05
    image = image.cpu() + noise
    return image.numpy().astype(np.float32)

## Start watching a folder

Create a temp folder, write the first image, launch `IO.watch`, then display the widget. New images will appear automatically.

In [ ]:
watch_dir = tempfile.mkdtemp(prefix="io_watch_demo_")
print(f"Watch folder: {watch_dir}")
# Write the first "capture"
img0 = make_lattice_image(size=512, defocus=0.0, seed=0)
np.save(f"{watch_dir}/capture_000.npy", img0)
# Create widget and start watching
w = Show2D(img0, title="Live Acquisition", pixel_size=1.2, show_fft=True, log_scale=True)
watcher = IO.watch(watch_dir, w, pattern="*.npy", interval=1.0)
print(f"Watcher: {watcher}")
w

## Simulate incoming data

Write new images to the watch folder every 2 seconds, simulating Velox writing EMD files during a focal series. Watch the gallery above grow in real time.

In [ ]:
def simulate_acquisition(folder, n_images=5, delay=2.0):
    """Write synthetic images to folder on a timer, simulating live capture."""
    defocus_values = np.linspace(-2, 2, n_images)
    for i, df in enumerate(defocus_values):
        time.sleep(delay)
        img = make_lattice_image(size=512, defocus=df, seed=i + 100)
        path = f"{folder}/capture_{i+1:03d}.npy"
        np.save(path, img)
        print(f"  Wrote capture_{i+1:03d}.npy (defocus={df:+.1f})")
    print("Acquisition complete.")
# Run in background so the notebook stays responsive
acq_thread = threading.Thread(
    target=simulate_acquisition,
    args=(watch_dir,),
    kwargs={"n_images": 5, "delay": 2.0},
    daemon=True,
)
acq_thread.start()
print("Simulating acquisition... watch the gallery above!")

## Check status

In [ ]:
print(f"Files loaded: {watcher.n_files}")
print(f"Images in gallery: {watcher.n_images}")
print(f"Running: {watcher.running}")
for f in watcher.files:
    print(f"  {f.name}")

## Mixed image sizes

`IO.watch` handles images of different sizes. Show2D auto-resizes all images to the largest dimensions.

In [ ]:
mixed_dir = tempfile.mkdtemp(prefix="io_watch_mixed_")
# Write images with different sizes (simulating overview + detail captures)
np.save(f"{mixed_dir}/overview_256.npy", make_lattice_image(size=256, defocus=0.0, seed=10))
np.save(f"{mixed_dir}/detail_1024.npy", make_lattice_image(size=1024, defocus=-0.5, seed=20))
w2 = Show2D(np.zeros((4, 4), dtype=np.float32), title="Mixed Sizes")
watcher2 = IO.watch(mixed_dir, w2, pattern="*.npy", interval=1.0)
print(f"Gallery: {w2.n_images} images, {w2.height}×{w2.width} (auto-resized to largest)")
w2

## Show3D: live time-series stack

`IO.watch` also works with Show3D — new files become frames in a playable stack. This is the natural mode for drift correction and dose-damage monitoring: each new capture appends as a frame, and you can scrub through the stack or hit play to animate.

In [ ]:
stack_dir = tempfile.mkdtemp(prefix="io_watch_stack_")
# Write first frame
np.save(f"{stack_dir}/frame_000.npy", make_lattice_image(size=512, defocus=-1.0, seed=200))
w3 = Show3D(np.random.rand(1, 64, 64).astype(np.float32), title="Live Time-Series")
watcher3 = IO.watch(stack_dir, w3, pattern="*.npy", interval=1.0)
w3

In [ ]:
# Simulate a drift series: 6 frames arriving every 2 seconds
def simulate_drift_series(folder, n_frames=6, delay=2.0):
    """Write frames with increasing drift, simulating a time-series at the microscope."""
    for i in range(n_frames):
        time.sleep(delay)
        # Each frame has slightly different defocus (simulating drift)
        df = -1.0 + i * 0.4
        img = make_lattice_image(size=512, defocus=df, seed=200 + i + 1)
        np.save(f"{folder}/frame_{i+1:03d}.npy", img)
        print(f"  Wrote frame_{i+1:03d}.npy (defocus={df:+.1f})")
    print("Drift series complete.")
drift_thread = threading.Thread(
    target=simulate_drift_series,
    args=(stack_dir,),
    kwargs={"n_frames": 6, "delay": 2.0},
    daemon=True,
)
drift_thread.start()
print("Simulating drift series... scrub through frames in the stack above!")

## vmin/vmax for A/B comparison

Use absolute `vmin`/`vmax` so all gallery images share the same intensity scale — essential for comparing raw vs processed, or images from different defocus settings.

In [ ]:
# Without vmin/vmax — each image auto-ranges to its own min/max (misleading)
raw = make_lattice_image(size=512, defocus=0.0, seed=50)
processed = raw * 0.5 + 0.1  # simulated processing: scaled + offset
Show2D([raw, processed], labels=["Raw", "Processed"], title="Without vmin/vmax (misleading)")

In [ ]:
# With vmin/vmax — both images use the same absolute scale (correct comparison)
Show2D([raw, processed], labels=["Raw", "Processed"],
       title="With vmin/vmax (correct)",
       vmin=float(raw.min()), vmax=float(raw.max()))

## Publication-quality export

`save_image()` now supports `title`, `colorbar`, and `scalebar` for publication-ready figures from Python.

In [ ]:
export_dir = tempfile.mkdtemp(prefix="io_watch_export_")
w_pub = Show2D(raw, title="HRTEM Lattice", pixel_size=1.2, cmap="inferno")
# Raw pixel export (fast, exact)
w_pub.save_image(f"{export_dir}/raw.png")
# Publication figure with title + colorbar + scale bar
out = w_pub.save_image(f"{export_dir}/figure.pdf", title=True, colorbar=True, scalebar=True, dpi=300)
print(f"Saved: {out} ({out.stat().st_size / 1024:.0f} KB)")

## Cleanup

Stop all watchers when done.

In [ ]:
watcher.stop()
watcher2.stop()
watcher3.stop()
print("All watchers stopped.")